# 汇总数据
`Aggregate`查询处理结果集以返回计算结果。使用`aggregate`查询来查询对象组或整个结果集。

In [1]:
import json
import weaviate
from weaviate.auth import AuthApiKey

# 连接到本地部署的 Weaviate
client = weaviate.Client(
    url="http://127.0.0.1:8080",
    auth_client_secret=AuthApiKey("WVF5YThaHlkYwhGUSmCRgsX3tD5ngdN8pkih")
)

In [2]:
print(client.is_ready())

True


## 检索count元属性
返回查询匹配的对象的数量。

In [3]:
response = (
    client.query
    .aggregate("JeopardyQuestion")
    .with_meta_count()
    .do()
)

print(json.dumps(response, indent=2))

{
  "data": {
    "Aggregate": {
      "JeopardyQuestion": [
        {
          "meta": {
            "count": 4
          }
        }
      ]
    }
  }
}


In [ ]:
### V4

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.aggregate.over_all(total_count=True)

print(response.total_count)

## 聚合text属性

此示例计算`question`属性中的出现频率：

In [4]:
response = (
    client.query
    .aggregate("JeopardyQuestion")
    .with_fields("answer { count type topOccurrences (limit: 5) { occurs value } }")  # `limit` here sets a minimum count threshold
    .do()
)

print(json.dumps(response, indent=2))

{
  "data": {
    "Aggregate": {
      "JeopardyQuestion": [
        {
          "answer": {
            "count": 3,
            "topOccurrences": [
              {
                "occurs": 2,
                "value": "Weaviate"
              },
              {
                "occurs": 1,
                "value": "Replaced"
              }
            ],
            "type": "text"
          }
        }
      ]
    }
  }
}


In [ ]:
from weaviate.classes.query import Metrics

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.aggregate.over_all(
    return_metrics=Metrics("answer").text(
        top_occurrences_count=True,
        top_occurrences_value=True,
        min_occurrences=5  # Threshold minimum count
    )
)

print(response.properties["answer"].top_occurrences)

## 聚合int属性
此示例总结了该points属性。

In [5]:
response = (
    client.query
    .aggregate("JeopardyQuestion")
    .with_fields("points { count sum }")
    .do()
)
print(json.dumps(response, indent=2))

{
  "errors": [
    {
      "locations": [
        {
          "column": 29,
          "line": 1
        }
      ],
      "message": "Cannot query field \"points\" on type \"AggregateJeopardyQuestion\".",
      "path": null
    }
  ]
}


In [ ]:
from weaviate.classes.query import Metrics

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.aggregate.over_all(
    # Use `.number` for floats (`NUMBER` datatype in Weaviate)
    return_metrics=Metrics("points").integer(sum_=True, maximum=True, minimum=True),
)

print(response.properties["points"].sum_)
print(response.properties["points"].minimum)
print(response.properties["points"].maximum)

## 聚合groupedBy属性

要对结果进行分组，请`groupBy`在查询中使用。

要检索每个组的聚合数据，请使用`groupedBy`属性。

In [ ]:
response = (
    client.query
    .aggregate("JeopardyQuestion")
    .with_group_by_filter(["round"])
    .with_fields("groupedBy { value }")
    .with_meta_count()
    .do()
)
print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.aggregate import GroupByAggregate

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.aggregate.over_all(
    group_by=GroupByAggregate(prop="round")
)

# print rounds names and the count for each
for group in response.groups:
    print(f"Value: {group.grouped_by.value} Count: {group.total_count}")

## 聚合similarity search
您可以`Aggregate`与相似性搜索运算符（运算符之一Near）一起使用。

用于`objectLimit`指定要聚合的最大对象数。

In [7]:
response = (
    client.query
    .aggregate("JeopardyQuestion")
    .with_near_text({
        "concepts": ["animals in space"]
    })
    .with_object_limit(10)
    .with_fields("points { sum }")
    .do()
)
print(json.dumps(response, indent=2))

{
  "errors": [
    {
      "locations": [
        {
          "column": 29,
          "line": 1
        }
      ],
      "message": "Unknown argument \"nearText\" on field \"JeopardyQuestion\" of type \"AggregateObjectsObj\". Did you mean \"nearVector\" or \"nearObject\"?",
      "path": null
    },
    {
      "locations": [
        {
          "column": 89,
          "line": 1
        }
      ],
      "message": "Cannot query field \"points\" on type \"AggregateJeopardyQuestion\".",
      "path": null
    }
  ]
}


In [ ]:
from weaviate.classes.query import Metrics

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.aggregate.near_text(
    query="animals in space",
    object_limit=10,
    return_metrics=Metrics("points").number(sum_=True),
)

print(response.properties["points"].sum_)

### 设置相似度distance
您可以`Aggregate`与相似性搜索运算符（运算符之一Near）一起使用。

用于`distance`指定对象的相似程度。

In [ ]:
response = (
    client.query
    .aggregate("JeopardyQuestion")
    .with_near_text({
        "concepts": ["animals in space"],
        "distance": 0.19
    })
    .with_fields("points { sum }")
    .do()
)

print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Metrics

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.aggregate.near_text(
    query="animals in space",
    distance=0.19,
    return_metrics=Metrics("points").number(sum_=True),
)

print(response.properties["points"].sum_)

## 聚合hybrid search
您可以与混合搜索Aggregate运算符一起使用。


In [ ]:
response = (
    client.query
    .aggregate("JeopardyQuestion")
    .with_hybrid({
        "query": "animals in space"
    })
    .with_object_limit(10)
    .with_fields("points { sum }")
    .do()
)
print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Metrics

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.aggregate.hybrid(
    query="animals in space",
    object_limit=10,
    return_metrics=Metrics("points").number(sum_=True),
)

print(response.properties["points"].sum_)

## 筛选结果
要获得更具体的结果，请使用filter缩小搜索范围。

In [ ]:
response = (
    client.query
    .aggregate("JeopardyQuestion")
    .with_where({
        "path": ["round"],
        "operator": "Equal",
        "valueText": "Final Jeopardy!"
    })
    .with_meta_count()
    .do()
)

print(json.dumps(response, indent=2))

In [ ]:
from weaviate.classes.query import Filter

jeopardy = client.collections.get("JeopardyQuestion")
response = jeopardy.aggregate.over_all(
    filters=Filter.by_property("round").equal("Final Jeopardy!"),
)

print(response.total_count)